# **05. Production Model and Insights**

In [49]:
# Import Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sidetable
import sklearn
import feature_engine
import scipy
from scipy import stats
from pathlib import Path
import pickle
import joblib
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, RobustScaler, OneHotEncoder, FunctionTransformer, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge, Lasso, ElasticNet, LinearRegression
from sklearn.metrics import make_scorer, mean_squared_error, r2_score

%matplotlib inline
sns.set_style('darkgrid')


In [2]:
# Display Settings
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)


In [3]:
# define file path for processed data
parent_path = Path.cwd().parent
tuned_model_file = parent_path.joinpath("models", "final_tuned_model.pkl")
processed_data_file = parent_path.joinpath("models", "processed.pkl")
original_data_file = parent_path.joinpath("data", "raw")
files = []
for file in original_data_file.rglob("*.csv"):
    files.append(file)
    print(file.name)
    print(files.index(file), " ", file)



sample_submission.csv
0   e:\house_price_prediction\Real-Estate-Price-Prediction\data\raw\extracted\sample_submission.csv
test.csv
1   e:\house_price_prediction\Real-Estate-Price-Prediction\data\raw\extracted\test.csv
train.csv
2   e:\house_price_prediction\Real-Estate-Price-Prediction\data\raw\extracted\train.csv


In [4]:
# Load necessary data and model
print("Loading necessary data and model...")
try:
    # Load the final tuned model from previous notebook
    with open(tuned_model_file, 'rb') as f:
        tuned_model = joblib.load(f)

    # Load processed data
    with open(processed_data_file, 'rb') as f:
        processed_data = joblib.load(f)

        # Extract features and target variable
        X_train = processed_data["X_train"]
        y_train = processed_data["y_train"]
        X_test = processed_data["X_test"]
        numeric_feat = processed_data["numeric_features"]
        ordinal_feat = processed_data["ordinal_features"]
        categorical_feat = processed_data["categorical_features"]
        boolean_feat = processed_data["boolean_features"]
        year_feat = processed_data["year_features"]
        test_id = processed_data['test_id']

    # Load original data for reference and insights
    train_orginial = pd.read_csv(files[2])

    # standardize the column names
    train_orginial.columns = train_orginial.columns.str.strip().str.replace(" ", "_").str.replace(".", "_")

    print("Data and model load successfully...")
except Exception as e:
    print(f"ErrorRaised : {e}")

Loading necessary data and model...


Data and model load successfully...


## **1. Model Inspection and Documentation**

In [51]:
# Extract and document the model components
print("\n ==== MODEL DOCUMENTATION ====")

# check the model type
model_type = tuned_model.named_steps['model'].__class__.__name__
print(f"Model type : {model_type}")

# Get the hyperparameter if possible
if hasattr(tuned_model.named_steps['model'], 'alpha'):
    print(f"Alpha (regularization strength) : {tuned_model.named_steps['model'].alpha:.6f}")
if hasattr(tuned_model.named_steps['model'], 'l1_ratio'):
    print(f"L1 Ratio : {tuned_model.named_steps['model'].l1_ratio}")

# Document the preprocessing steps
print("\nPreprocessing Pipeline :")
for name, transformer, features in tuned_model.named_steps['preprocessor'].transformers_:
    if name == 'numeric':
        print(f" - Numeric Features ({len(numeric_feat)}) : {', '.join(numeric_feat[:5])}...")
        for step_name, step in transformer.steps:
            print(f"   * {step_name} : {step.__class__.__name__}")
    
    elif name == 'boolean':
        print(f"\n - Boolean Features ({len(boolean_feat)}) : {', '.join(boolean_feat[:5])}...")
        for step_name, step in transformer.steps:
            print(f"   * {step_name} : {step.__class__.__name__}")
    
    elif name == 'ordinal':
        print(f"\n - Ordinal Features ({len(ordinal_feat)}) : {', '.join(ordinal_feat[:5])}...")
        for step_name, step in transformer.steps:
            print(f"   * {step_name} : {step.__class__.__name__}")
    
    elif name == 'year':
        print(f"\n - Year Features ({len(year_feat)}) : {', '.join(year_feat[:5])}...")
        for step_name, step in transformer.steps:
            print(f"   * {step_name} : {step.__class__.__name__}")
        
    elif name == 'cat':
        print(f"\n - Categorical Features ({len(categorical_feat)}) : {", ".join(categorical_feat)}")
        for step_name, step in transformer.steps:
            print(f"   * {step_name} : {step.__class__.__name__}")

# Model performance metrics
print("\nModel Performance :")
y_pred = tuned_model.predict(X_train)
rmse = np.sqrt(mean_squared_error(y_train, y_pred))
r2 = r2_score(y_train, y_pred)
    
print(f" - RMSE on training data: ${rmse:.2f}")
print(f" - R² on training data: {r2:.4f}")


 ==== MODEL DOCUMENTATION ====
Model type : ElasticNet
Alpha (regularization strength) : 0.016681
L1 Ratio : 0.9

Preprocessing Pipeline :
 - Numeric Features (9) : LivAreaQual, TotalSF, NeighborhoodMedianPrice, HouseAge, QualCond...
   * imputer : SimpleImputer
   * scaler : StandardScaler

 - Boolean Features (4) : HasFireplace, HasGarage, HasPorch, CentralAir...
   * bool : OrdinalEncoder

 - Ordinal Features (8) : ExterQual, KitchenQual, BsmtQual, GarageFinish, OverallQual...
   * scaler : StandardScaler

 - Year Features (1) : GarageYrBlt...
   * passthrough : str

 - Categorical Features (1) : Neighborhood
   * imputer : SimpleImputer
   * onehot : OneHotEncoder

Model Performance :
 - RMSE on training data: $32746.17
 - R² on training data: 0.8301
